In [1]:
import pandas as pd
import numpy as np

In [2]:
# File path (as provided)
data_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_processed.csv"

# Load dataset
df = pd.read_csv(data_path)

# Quick sanity check
df.shape, df.head(3)

((117, 120),
    BUILDING_REFERENCE_NUMBER  OSG_REFERENCE_NUMBER              ADDRESS1  \
 0                 1001829067           122009933.0  19 CULCREUCH AVENUE    
 1                 1001951858           122010025.0             GLENVIEW    
 2                 1001829071           122009868.0  22 CULCREUCH AVENUE    
 
   ADDRESS2  ADDRESS3 POSTCODE   LATITUDE  LONGITUDE INSPECTION_DATE  \
 0  FINTRY   GLASGOW   G63 0YB  56.055692  -4.223337         9/29/25   
 1  FINTRY   GLASGOW   G63 0XJ  56.052824  -4.220640         9/19/25   
 2  FINTRY   GLASGOW   G63 0YB  56.055503  -4.223691         7/29/25   
 
          TYPE_OF_ASSESSMENT  ...  \
 0  RdSAP, existing dwelling  ...   
 1  RdSAP, existing dwelling  ...   
 2  RdSAP, existing dwelling  ...   
 
                              WALL_DESCRIPTION_PARSED  \
 0  [{'description': 'Cavity wall, filled cavity',...   
 1  [{'description': 'Timber frame, as built, insu...   
 2  [{'description': 'Cavity wall, filled cavity',...   
 
       

In [3]:
required_columns = [
    "CURRENT_ENERGY_EFFICIENCY",
    "POTENTIAL_ENERGY_EFFICIENCY",
    "ENVIRONMENT_IMPACT_CURRENT",
    "ENVIRONMENT_IMPACT_POTENTIAL",
    "ENERGY_CONSUMPTION_CURRENT",
    "ENERGY_CONSUMPTION_POTENTIAL",
    "CO2_EMISSIONS_CURRENT",
    "CO2_EMISSIONS_POTENTIAL",
    "3_YR_ENERGY_COST_CURRENT",
    "3_YR_ENERGY_SAVINGS_POTENTIAL"
]

missing = [c for c in required_columns if c not in df.columns]

missing

[]

In [4]:
numeric_cols = [
    "CURRENT_ENERGY_EFFICIENCY",
    "POTENTIAL_ENERGY_EFFICIENCY",
    "ENVIRONMENT_IMPACT_CURRENT",
    "ENVIRONMENT_IMPACT_POTENTIAL",
    "ENERGY_CONSUMPTION_CURRENT",
    "ENERGY_CONSUMPTION_POTENTIAL",
    "CO2_EMISSIONS_CURRENT",
    "CO2_EMISSIONS_POTENTIAL",
    "3_YR_ENERGY_COST_CURRENT",
    "3_YR_ENERGY_SAVINGS_POTENTIAL"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [5]:
df["DELTA_ENERGY_EFFICIENCY"] = (
    df["POTENTIAL_ENERGY_EFFICIENCY"] -
    df["CURRENT_ENERGY_EFFICIENCY"]
)

In [6]:
df["DELTA_ENVIRONMENT_IMPACT"] = (
    df["ENVIRONMENT_IMPACT_POTENTIAL"] -
    df["ENVIRONMENT_IMPACT_CURRENT"]
)

In [7]:
df["DELTA_ENERGY_CONSUMPTION"] = (
    df["ENERGY_CONSUMPTION_CURRENT"] -
    df["ENERGY_CONSUMPTION_POTENTIAL"]
)

df["PCT_ENERGY_REDUCTION"] = (
    df["DELTA_ENERGY_CONSUMPTION"] /
    df["ENERGY_CONSUMPTION_CURRENT"]
)

In [8]:
df["DELTA_CO2_EMISSIONS"] = (
    df["CO2_EMISSIONS_CURRENT"] -
    df["CO2_EMISSIONS_POTENTIAL"]
)

df["PCT_CO2_REDUCTION"] = (
    df["DELTA_CO2_EMISSIONS"] /
    df["CO2_EMISSIONS_CURRENT"]
)

In [9]:
df["ANNUAL_ENERGY_COST_SAVINGS"] = (
    df["3_YR_ENERGY_SAVINGS_POTENTIAL"] / 3
)

In [10]:
df["HAS_POTENTIAL_DATA"] = (
    df["POTENTIAL_ENERGY_EFFICIENCY"].notna() &
    df["ENERGY_CONSUMPTION_POTENTIAL"].notna()
)

df["POSITIVE_ENERGY_SAVINGS"] = df["DELTA_ENERGY_CONSUMPTION"] > 0
df["POSITIVE_CO2_SAVINGS"] = df["DELTA_CO2_EMISSIONS"] > 0

In [11]:
summary_cols = [
    "DELTA_ENERGY_EFFICIENCY",
    "DELTA_ENVIRONMENT_IMPACT",
    "DELTA_ENERGY_CONSUMPTION",
    "PCT_ENERGY_REDUCTION",
    "DELTA_CO2_EMISSIONS",
    "ANNUAL_ENERGY_COST_SAVINGS"
]

df[summary_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
DELTA_ENERGY_EFFICIENCY,117.0,22.658120,16.004936,0.0,11.000000,20.000000,31.000000,70.000000
DELTA_ENVIRONMENT_IMPACT,117.0,19.290598,14.033249,0.0,10.000000,17.000000,26.000000,64.000000
DELTA_ENERGY_CONSUMPTION,117.0,136.008547,104.915524,0.0,68.000000,112.000000,179.000000,476.000000
PCT_ENERGY_REDUCTION,117.0,0.473298,0.320104,0.0,0.287234,0.426295,0.597403,2.104167
DELTA_CO2_EMISSIONS,117.0,3.412821,2.626310,-0.8,1.400000,3.000000,4.900000,15.000000
ANNUAL_ENERGY_COST_SAVINGS,117.0,594.692308,703.676092,0.0,127.000000,324.000000,806.000000,3755.000000


In [12]:
output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_step1_improvement_metrics.csv"

df.to_csv(output_path, index=False)

output_path

'/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_step1_improvement_metrics.csv'